# Notebook 06: Graph Construction

This notebook converts the training splits from Notebook 05 into the bipartite
graphs required by the recommendation models.

Users and items share a single node index space, producing one square symmetric
adjacency matrix. Graphs are built **only** from the training split to prevent
information leakage from the validation or test sets.

Only the edge index is stored. The normalized adjacency is inexpensive to
rebuild when needed, avoiding large derived files that could become inconsistent
with the saved splits.

## 1. Imports and configuration

Graphs are built for every subset and protocol. `small` is the one the main experiments
use, but the scalability runs need `medium` and `full`, and the cold-start experiment
needs `small/timestamp`.

In [2]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Paths
SUBSETS_DIR = PROJECT_ROOT / "data" / "processed" / "subsets_earliest"
GRAPHS_DIR = PROJECT_ROOT / "data" / "processed" / "graphs_earliest"
FIGURES_DIR = PROJECT_ROOT / "figures"

GRAPHS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Config
SUBSET_LEVELS = ["small", "medium", "full"]
PROTOCOLS = ["last_out", "timestamp"]
TARGETS = [(s, p) for s in SUBSET_LEVELS for p in PROTOCOLS]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"torch {torch.__version__}, device: {DEVICE}")
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"{props.name}, {props.total_memory / 1e9:.1f} GB, sm_{props.major}{props.minor}")
print(f"\nbuilding {len(TARGETS)} graphs: {TARGETS}")

torch 2.11.0+cu128, device: cuda
NVIDIA GeForce RTX 5070 Ti Laptop GPU, 12.8 GB, sm_120

building 6 graphs: [('small', 'last_out'), ('small', 'timestamp'), ('medium', 'last_out'), ('medium', 'timestamp'), ('full', 'last_out'), ('full', 'timestamp')]


## 2. Load the splits

Training interactions are used to construct the graph, while the validation and
test splits are loaded only for leakage checks.

The number of users and items comes from the mapping files rather than the
training split, since some nodes may have no training interactions under the
timestamp protocol but still require embeddings.

In [3]:
def load_target(subset, protocol):
    base = SUBSETS_DIR / subset
    splits_dir = base / "splits" / protocol

    parts = {
        name: pq.read_table(splits_dir / f"{name}.parquet",
                            columns=["user_idx", "item_idx"]).to_pandas()
        for name in ["train", "validation", "test"]
    }
    return {
        "n_users": len(pq.read_table(base / "user_mapping.parquet")),
        "n_items": len(pq.read_table(base / "item_mapping.parquet")),
        **parts,
    }


loaded = {}
for subset, protocol in TARGETS:
    t0 = time.time()
    loaded[(subset, protocol)] = load_target(subset, protocol)
    d = loaded[(subset, protocol)]
    print(f"{subset:>6} {protocol:>10}  "
          f"{d['n_users']:>9,} users  {d['n_items']:>8,} items  "
          f"train {len(d['train']):>10,}  ({time.time()-t0:.1f}s)")

 small   last_out    258,597 users   105,298 items  train  1,829,582  (0.4s)
 small  timestamp    258,597 users   105,298 items  train  2,020,041  (0.1s)
medium   last_out    676,170 users   211,842 items  train  4,945,119  (0.1s)
medium  timestamp    676,170 users   211,842 items  train  5,358,564  (0.1s)
  full   last_out  1,641,026 users   368,228 items  train 12,191,484  (0.2s)
  full  timestamp  1,641,026 users   368,228 items  train 13,096,427  (0.1s)


## 3. Build the graphs

Each interaction becomes two directed edges, producing a symmetric adjacency.
Item node indices are offset by `n_users` so users and items occupy the same
node index space.

Only the edge index is created here. The normalized adjacency is derived from it
when required.

In [4]:
from src import graph as gr

graphs = {}
stat_rows = []

for subset, protocol in TARGETS:
    d = loaded[(subset, protocol)]
    t0 = time.time()

    edge_index = gr.build_edge_index(d["train"], d["n_users"], d["n_items"])
    graphs[(subset, protocol)] = edge_index

    stats = gr.graph_statistics(edge_index, d["n_users"], d["n_items"])
    stat_rows.append({"subset": subset, "protocol": protocol, **stats})

    size_mb = edge_index.element_size() * edge_index.numel() / 1e6
    print(f"{subset:>6} {protocol:>10}  "
          f"{stats['nodes']:>9,} nodes  {stats['directed_edges']:>11,} directed edges  "
          f"{size_mb:>6.0f} MB  ({time.time()-t0:.1f}s)")

graph_stats = pd.DataFrame(stat_rows)
graph_stats

 small   last_out    363,895 nodes    3,659,164 directed edges      59 MB  (0.1s)
 small  timestamp    363,895 nodes    4,040,082 directed edges      65 MB  (0.1s)
medium   last_out    888,012 nodes    9,890,238 directed edges     158 MB  (0.2s)
medium  timestamp    888,012 nodes   10,717,128 directed edges     171 MB  (0.2s)
  full   last_out  2,009,254 nodes   24,382,968 directed edges     390 MB  (0.4s)
  full  timestamp  2,009,254 nodes   26,192,854 directed edges     419 MB  (0.5s)


,subset,protocol,nodes,users,items,interactions,directed_edges,isolated_nodes,mean_user_degree,mean_item_degree,min_user_degree,min_item_degree,max_user_degree,max_item_degree
0,small,last_out,363895,258597,105298,1829582,3659164,426,7.075032,17.375278,3.0,0.0,629.0,6779.0
1,small,timestamp,363895,258597,105298,2020041,4040082,13436,7.811541,19.184040,0.0,0.0,631.0,7937.0
2,medium,last_out,888012,676170,211842,4945119,9890238,692,7.313426,23.343430,3.0,0.0,833.0,16393.0
3,medium,timestamp,888012,676170,211842,5358564,10717128,35966,7.924877,25.295097,0.0,0.0,784.0,19156.0
4,full,last_out,2009254,1641026,368228,12191484,24382968,1176,7.429184,33.108521,3.0,0.0,936.0,38518.0
5,full,timestamp,2009254,1641026,368228,13096427,26192854,74991,7.980633,35.566082,0.0,0.0,852.0,44837.0


## 4. Normalise and verify

The normalized adjacency follows the standard LightGCN formulation
`D^-1/2 A D^-1/2` without self-loops.

Each graph is verified by checking:

- symmetry
- node degrees
- spectral radius
- absence of validation and test edges

The normalized adjacency is built only for verification and is reconstructed by
the models when needed.

In [14]:
# I ran this anytime I made a change to src/graph.py
import importlib
from src import graph as gr
importlib.reload(gr)

<module 'src.graph' from 'c:\\Projects\\gnn-recommender-dissertation\\src\\graph.py'>

In [9]:
verify_rows = []

for subset, protocol in TARGETS:
    d = loaded[(subset, protocol)]
    edge_index = graphs[(subset, protocol)]
    n_nodes = d["n_users"] + d["n_items"]
    t0 = time.time()

    # Build the normalized adjacency used by LightGCN:
    # D^-1/2 A D^-1/2
    adj = gr.normalised_adjacency(edge_index, n_nodes, device=DEVICE)

    # Estimate the spectral radius by power iteration. For a correctly
    # normalized adjacency this should converge to 1.
    radius, iterations = gr.spectral_radius(adj)

    deg = gr.node_degrees(edge_index, n_nodes)

    # Training interaction counts provide the expected user degrees.
    train_user_counts = torch.bincount(
        torch.from_numpy(d["train"]["user_idx"].to_numpy().astype(np.int64)),
        minlength=d["n_users"],
    ).to(torch.float32)

    verify_rows.append({
        "subset": subset,
        "protocol": protocol,

        # Every edge must have its reverse edge.
        "symmetric": gr.verify_symmetry(edge_index),

        # User degrees should match the training interaction counts.
        "degrees_match": bool(torch.equal(deg[:d["n_users"]], train_user_counts)),

        # Validation/test interactions must never appear in the training graph.
        "spectral_radius": round(radius, 7),
        "power_iterations": iterations,
        "validation_edges_in_graph": gr.count_held_out_edges(
            edge_index, d["validation"], d["n_users"]),
        "test_edges_in_graph": gr.count_held_out_edges(
            edge_index, d["test"], d["n_users"]),
        "nnz": int(adj._nnz()),
        "seconds": round(time.time() - t0, 1),
    })
    print(f"{subset:>6} {protocol:>10}  radius {radius:.7f} "
          f"in {iterations} iters  ({time.time()-t0:.0f}s)")

    del adj
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

verification = pd.DataFrame(verify_rows)
verification

 small   last_out  radius 0.9999998 in 43 iters  (1s)
 small  timestamp  radius 0.9999998 in 31 iters  (1s)
medium   last_out  radius 0.9999996 in 56 iters  (3s)
medium  timestamp  radius 0.9999998 in 35 iters  (3s)
  full   last_out  radius 0.9999989 in 93 iters  (7s)
  full  timestamp  radius 0.9999986 in 60 iters  (7s)


,subset,protocol,symmetric,degrees_match,spectral_radius,power_iterations,validation_edges_in_graph,test_edges_in_graph,nnz,seconds
0,small,last_out,True,True,1.000000,43,0,0,3659164,1.3
1,small,timestamp,True,True,1.000000,31,0,0,4040082,1.3
2,medium,last_out,True,True,1.000000,56,0,0,9890238,3.3
3,medium,timestamp,True,True,1.000000,35,0,0,10717128,3.3
4,full,last_out,True,True,0.999999,93,0,0,24382968,6.9
5,full,timestamp,True,True,0.999999,60,0,0,26192854,7.2


In [10]:
assert verification["symmetric"].all(), "adjacency is not symmetric"
assert verification["degrees_match"].all(), "degrees do not match training counts"
assert (verification["validation_edges_in_graph"] == 0).all(), "validation edges leaked into graph"
assert (verification["test_edges_in_graph"] == 0).all(), "test edges leaked into graph"

# power iteration converges upward to 1, so anything meaningfully above it would be
# a bug rather than a slow run
assert (verification["spectral_radius"] <= 1 + 1e-4).all(), "spectral radius above 1"
assert (verification["spectral_radius"] >= 1 - 1e-4).all(), \
    "spectral radius below 1 - either not converged or the normalisation is wrong"

print("all graphs verified: symmetric, correct degrees, no held-out edges")

all graphs verified: symmetric, correct degrees, no held-out edges


## 5. Connectivity

How fragmented the graph is matters for the GNNs. Information only propagates within a
connected component, so if the graph is one large component every user can in principle
be reached from every item a few hops away. If it's split into many pieces, some users
are permanently cut off from most of the catalogue no matter how many layers the model
has.

Isolated nodes count as their own single-node components, so the number under the
timestamp protocol will be at least the number of cold users and cold items.

In [13]:
component_rows = []

for subset, protocol in TARGETS:
    d = loaded[(subset, protocol)]
    n_nodes = d["n_users"] + d["n_items"]
    t0 = time.time()

    edge_index = graphs[(subset, protocol)].to(DEVICE)
    labels = gr.connected_components(edge_index, n_nodes)

    sizes = torch.bincount(labels)
    sizes = sizes[sizes > 0]
    largest = int(sizes.max())

    component_rows.append({
        "subset": subset,
        "protocol": protocol,
        "components": int(sizes.numel()),
        "largest_component": largest,
        "share_in_largest": round(largest / n_nodes, 4),
        "singletons": int((sizes == 1).sum()),
        "seconds": round(time.time() - t0, 1),
    })
    print(f"{subset:>6} {protocol:>10}  {int(sizes.numel()):>7,} components  "
          f"largest covers {largest / n_nodes:.2%}  ({time.time()-t0:.0f}s)")

    del edge_index, labels
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

components = pd.DataFrame(component_rows)
components

 small   last_out      427 components  largest covers 99.88%  (0s)
 small  timestamp   13,458 components  largest covers 96.30%  (0s)
medium   last_out      693 components  largest covers 99.92%  (0s)
medium  timestamp   35,996 components  largest covers 95.94%  (0s)
  full   last_out    1,177 components  largest covers 99.94%  (0s)
  full  timestamp   75,027 components  largest covers 96.26%  (0s)


,subset,protocol,components,largest_component,share_in_largest,singletons,seconds
0,small,last_out,427,363469,0.9988,426,0.2
1,small,timestamp,13458,350416,0.9630,13436,0.1
2,medium,last_out,693,887320,0.9992,692,0.1
3,medium,timestamp,35996,851987,0.9594,35966,0.0
4,full,last_out,1177,2008078,0.9994,1176,0.1
5,full,timestamp,75027,1934193,0.9626,74991,0.1


## 6. Save

Only graph statistics are written to disk. The edge index is rebuilt directly
from `train.parquet`, making it unnecessary to store large derived graph files.

The loader below verifies that rebuilding the graph reproduces the same result.

In [15]:
# the loader has to reproduce what we built above, or the models would be
# training on something different from what was verified
for subset, protocol in TARGETS:
    rebuilt = gr.load_graph(SUBSETS_DIR, subset, protocol)
    assert torch.equal(rebuilt["edge_index"], graphs[(subset, protocol)])
    assert rebuilt["n_users"] == loaded[(subset, protocol)]["n_users"]
    assert rebuilt["n_items"] == loaded[(subset, protocol)]["n_items"]

print(f"load_graph reproduces all {len(TARGETS)} graphs exactly")

load_graph reproduces all 6 graphs exactly


In [16]:
summary = {
    "notebook": "06_graph_construction",
    "source": str(SUBSETS_DIR.relative_to(PROJECT_ROOT)),
    "node_layout": "users 0..U-1, items U..U+I-1",
    "normalisation": "D^-1/2 A D^-1/2, no self-loops",
    "storage": "edge index derived on demand via graph.load_graph()",
    "statistics": graph_stats.to_dict(orient="records"),
    "verification": verification.to_dict(orient="records"),
    "components": components.to_dict(orient="records"),
}

summary_path = GRAPHS_DIR / "graph_summary.json"
with open(summary_path, "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, default=str)

print(f"summary -> {summary_path.relative_to(PROJECT_ROOT)}")

summary -> data\processed\graphs_earliest\graph_summary.json


## 7. Final checks

Everything was checked as it was built. This gathers it into one cell, and adds a
cross-check against the Notebook 05 summary so a mismatch between the two notebooks
would show up here rather than during training.

In [17]:
with open(SUBSETS_DIR / "subsets_summary.json", encoding="utf-8") as fh:
    subsets_summary = json.load(fh)

nb05_subsets = {r["subset"]: r for r in subsets_summary["subsets"]}
nb05_coverage = {(r["subset"], r["protocol"]): r for r in subsets_summary["coverage"]}
nb05_splits = {(r["subset"], r["protocol"]): r for r in subsets_summary["splits"]}

checks = []

def check(label, condition):
    checks.append((label, bool(condition)))

for _, row in verification.iterrows():
    key = (row["subset"], row["protocol"])
    check(f"{key[0]}/{key[1]}: symmetric", row["symmetric"])
    check(f"{key[0]}/{key[1]}: degrees match training", row["degrees_match"])
    check(f"{key[0]}/{key[1]}: no validation edges", row["validation_edges_in_graph"] == 0)
    check(f"{key[0]}/{key[1]}: no test edges", row["test_edges_in_graph"] == 0)
    check(f"{key[0]}/{key[1]}: spectral radius is 1", abs(row["spectral_radius"] - 1) < 1e-4)

for _, row in graph_stats.iterrows():
    key = (row["subset"], row["protocol"])
    check(f"{key[0]}/{key[1]}: node count matches subset",
          row["users"] == nb05_subsets[key[0]]["users_kept"]
          and row["items"] == nb05_subsets[key[0]]["items"])
    check(f"{key[0]}/{key[1]}: edges = 2 x training interactions",
          row["directed_edges"] == 2 * nb05_splits[key]["train"])
    # isolated nodes should be at least the cold users plus cold items from NB05,
    # and can exceed it because NB05 only counted cold items that appear in test
    check(f"{key[0]}/{key[1]}: isolated nodes consistent with NB05",
          row["isolated_nodes"] >= nb05_coverage[key]["cold_users"]
                                  + nb05_coverage[key]["cold_items"])

for _, row in components.iterrows():
    check(f"{row['subset']}/{row['protocol']}: one dominant component",
          row["share_in_largest"] > 0.95)

width = max(len(l) for l, _ in checks)
for label, passed in checks:
    print(f"{'PASS' if passed else 'FAIL'}  {label:{width}s}")

assert all(p for _, p in checks), "some graph checks failed"
print(f"\nall {len(checks)} checks passed")

PASS  small/last_out: symmetric                            
PASS  small/last_out: degrees match training               
PASS  small/last_out: no validation edges                  
PASS  small/last_out: no test edges                        
PASS  small/last_out: spectral radius is 1                 
PASS  small/timestamp: symmetric                           
PASS  small/timestamp: degrees match training              
PASS  small/timestamp: no validation edges                 
PASS  small/timestamp: no test edges                       
PASS  small/timestamp: spectral radius is 1                
PASS  medium/last_out: symmetric                           
PASS  medium/last_out: degrees match training              
PASS  medium/last_out: no validation edges                 
PASS  medium/last_out: no test edges                       
PASS  medium/last_out: spectral radius is 1                
PASS  medium/timestamp: symmetric                          
PASS  medium/timestamp: degrees match tr

# Findings

**Training graphs are leakage-free.** No validation or test interactions appear
in any graph.

**Graph construction was verified successfully.** All adjacency matrices are
symmetric, node degrees match the training data, and every graph reached the
expected spectral radius.

**The graphs are highly connected.** Leave-last-out produces one giant component
plus isolated nodes, while the timestamp graphs remain approximately 96%
connected, with most remaining nodes corresponding to expected cold users and
cold items.

**Graphs are rebuilt rather than stored.** Reconstructing them from the training
splits is fast, avoids storing large derived files, and guarantees consistency
with the saved data.

**The graph sizes are suitable for the planned experiments.** The `small`
dataset comfortably fits local GPU memory, while the `full` dataset remains
manageable for scalability experiments.

# Generative AI use 

AI was used to clarify aspects of graph construction and connectivity and to suggest verification checks for graph symmetry, node degrees, normalisation and held-out edge leakage. These checks were implemented and run before proceeding to model training.